<a href="https://colab.research.google.com/github/victoriaktruong/integrative-embedding-breastTME/blob/main/Mixedbread_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This script preprocesses the primary breast tumor dataset from [Xu et al. (2024)](https://www.cell.com/cell-reports-medicine/fulltext/S2666-3791%2824%2900180-0), which contains 236,363 cells.

The code was written by [Emmett Peng](https://github.com/Emmett-Peng).

# Preprocessing

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers sentence-transformers --quiet
!pip install scanpy --quiet

In [3]:
import scanpy as sc
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from google.colab import files
import os
import matplotlib.pyplot as plt

In [4]:
file_path = '/content/drive/MyDrive/HuLab/gene_name_sequences.txt'

with open(file_path, 'r') as file:
    sequences = file.readlines()

# Process the sequences
sequences = [seq.strip() for seq in sequences if seq.strip()]
print(f"Loaded {len(sequences)} sequences")

Loaded 175942 sequences


#Generate cell embeddings using Mixedbread

The embedding generation cell below was not executed because sequence processing takes over 10 hours.

In [ ]:
# Set embedding size
desired_dimension = 768

# Load in mixedbread model
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", truncate_dim=desired_dimension)

# Generate embeddings for each sequence
embeddings = []
for i, sequence in enumerate(sequences):
    print(f"Processing sequence {i + 1}/{len(sequences)}")

    sequence_embedding = model.encode(sequence, normalize_embeddings=True)
    embeddings.append(sequence_embedding)

embeddings = np.array(embeddings)
print(f"Generated embeddings with shape: {embeddings.shape}")
np.save('embeddings_mxbai.npy', embeddings)

Loading in the generated embeddings:

In [5]:
embeddings = np.load("/content/drive/MyDrive/HuLab/embeddings_mxbai.npy")

# Load ordered cell types from the text file
cell_types_path = '/content/drive/MyDrive/HuLab/ordered_cell_types.txt'
with open(cell_types_path, 'r') as file:
    cell_types = [line.strip() for line in file]
if len(cell_types) != len(embeddings):
    raise ValueError("Number of cell types does not match the number of embeddings!")

In [6]:
# DataFrame with embeddings
embedding_dim = embeddings.shape[1]
columns = [f"embedding_{i+1}" for i in range(embedding_dim)]
df = pd.DataFrame(embeddings, columns=columns)
df['cell_type'] = cell_types

print(df.head(), df.shape)

   embedding_1  embedding_2  embedding_3  embedding_4  embedding_5  \
0     0.056387    -0.034330     0.023211     0.047154     0.002368   
1     0.048038    -0.035414     0.038727     0.038417    -0.015191   
2     0.054466    -0.029345     0.020432     0.034670     0.005207   
3     0.049267    -0.036468     0.029062     0.048696    -0.001872   
4     0.062300    -0.029875     0.021546     0.044744     0.012145   

   embedding_6  embedding_7  embedding_8  embedding_9  embedding_10  ...  \
0    -0.062339    -0.021944    -0.024997     0.013336      0.046091  ...   
1    -0.053206    -0.019493    -0.039178     0.002910      0.066409  ...   
2    -0.046743    -0.040258    -0.033733     0.022962      0.057154  ...   
3    -0.060754    -0.018023    -0.024259     0.005852      0.059082  ...   
4    -0.043419    -0.017427    -0.024588     0.007944      0.063959  ...   

   embedding_760  embedding_761  embedding_762  embedding_763  embedding_764  \
0       0.016347       0.025029      -0.09